In [1]:
import gzip
import json
import os
from datetime import datetime
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from tqdm import tqdm

from kibad_crawler.config import RAW_DATA_DIR, S2_API_KEY, MAX_NUM_BULK_PAPERS
from kibad_crawler import s2_api

2026-06-03 21:01:08.600 | INFO     | kibad_crawler.config:<module>:12 - PROJ_ROOT path is: /Users/darenasc/projects/dfki/kibad-crawler


In [2]:
s2_client = s2_api.S2Client()

2026-06-03 21:01:08.608 | DEBUG    | kibad_crawler.s2_api:get_releases:402 - Fetching releases: https://api.semanticscholar.org/datasets/v1/release
2026-06-03 21:01:08.912 | INFO     | kibad_crawler.s2_api:__init__:109 - Semantic Scholar API key found
2026-06-03 21:01:08.912 | INFO     | kibad_crawler.s2_api:__init__:110 - '2026-05-26' is the latest release


# Übersicht relevante Journals und Archive

## Publication Venues

Download [Übersicht relevante Journals und Archive](https://docs.google.com/spreadsheets/d/1ugPuiRNmcVYWcV98TvXsyqVwJYhmk2iZjZ4o9estM2Y/edit?gid=1647072097#gid=1647072097) as CSV and save it to `RAW_DATA_DIR`

In [4]:
df_venues = s2_client.get_publication_venues()

publication_venue_ids = s2_api.get_publication_venue_id_from_issn_codes(
    file_path=RAW_DATA_DIR
    / "Übersicht relevante Journals und Archive - Ergebnis systematische Literatursuche.csv",
    df_venues=df_venues,
)

2026-06-03 21:01:08.931 | DEBUG    | kibad_crawler.s2_api:get_dataset_url:162 - Acquiring links for dataset: release='2026-05-26' dataset='publication-venues'
2026-06-03 21:01:09.275 | INFO     | kibad_crawler.s2_api:get_dataset_url:168 - 1 files for '2026-05-26' 'publication-venues' dataset
2026-06-03 21:01:09.276 | DEBUG    | kibad_crawler.s2_api:download_file:224 - Downloading file: '20260529_125811_00178_xpgke_1510d09c-c8b2-4ea4-817b-29729d5f1c8b.gz' -> '/Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/release/publication_venues'
2026-06-03 21:01:12.739 | INFO     | kibad_crawler.s2_api:download_file:232 - Downloaded /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/release/publication_venues/20260529_125811_00178_xpgke_1510d09c-c8b2-4ea4-817b-29729d5f1c8b.gz 15.3 MiB
2026-06-03 21:01:13.587 | INFO     | kibad_crawler.s2_api:get_publication_venue_id_from_issn_codes:81 - 1230 journals in the file | 129 ISSN codes | 95 S2 'publ

# Semantic Scholar

## Papers

In [5]:
release_path = s2_client._download_all_papers(
    publication_venue_ids=publication_venue_ids, n_files=2, persistant=True
)

2026-06-03 21:01:13.593 | DEBUG    | kibad_crawler.s2_api:get_dataset_url:162 - Acquiring links for dataset: release='2026-05-26' dataset='papers'
2026-06-03 21:01:14.268 | INFO     | kibad_crawler.s2_api:get_dataset_url:168 - 60 files for '2026-05-26' 'papers' dataset
2026-06-03 21:01:14.268 | INFO     | kibad_crawler.s2_api:_download_all_papers:314 - Downloading 2 paper files to /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-05-26


  0%|          | 0/2 [00:00<?, ?it/s]

2026-06-03 21:01:14.270 | DEBUG    | kibad_crawler.s2_api:download_file:224 - Downloading file: '20260529_070610_00124_u6igq_002c5728-5592-4231-9055-b3b50c72068c.gz' -> '/Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-05-26/papers'


  0%|          | 0/2 [02:09<?, ?it/s]

2026-06-03 21:03:23.968 | INFO     | kibad_crawler.s2_api:download_file:232 - Downloaded /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-05-26/papers/20260529_070610_00124_u6igq_002c5728-5592-4231-9055-b3b50c72068c.gz 1.0 GiB


 50%|█████     | 1/2 [05:29<05:29, 329.76s/it]

... (3 outputs left out)

 50%|█████     | 1/2 [06:53<05:29, 329.76s/it]

2026-06-03 21:08:07.405 | INFO     | kibad_crawler.s2_api:download_file:232 - Downloaded /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-05-26/papers/20260529_070610_00124_u6igq_08555d5d-32fe-4398-9238-19aeb3d4574a.gz 612.5 MiB


100%|██████████| 2/2 [08:51<00:00, 265.76s/it]

2026-06-03 21:10:05.683 | INFO     | kibad_crawler.s2_api:download_paper_batch:274 - Wrote /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-05-26/papers/20260529_070610_00124_u6igq_08555d5d-32fe-4398-9238-19aeb3d4574a.gz.csv (5510 rows)
2026-06-03 21:10:05.729 | DEBUG    | kibad_crawler.s2_api:download_paper_batch:278 - Removed /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-05-26/papers/20260529_070610_00124_u6igq_08555d5d-32fe-4398-9238-19aeb3d4574a.gz


## Bulk paper download

In [7]:
paper_files = [x for x in (release_path).glob("*.csv")]
dfs = [pd.read_csv(f) for f in paper_files]
df_all_papers = pd.concat(dfs)
print(f"{len(df_all_papers):,} papers in total")

14,534 papers in total


In [8]:
df_all_papers.head()

,corpusid,externalids,url,title,authors,venue,publicationvenueid,year,referencecount,citationcount,influentialcitationcount,isopenaccess,s2fieldsofstudy,publicationtypes,publicationdate,journal
0,259646606,"{'MAG': None, 'CorpusId': '259646606', 'ACL': ...",https://www.semanticscholar.org/paper/9e57dccb...,Induction role of chitosan nanoparticles to An...,"[{'authorId': '50647309', 'name': 'A. Mashraqi'}]",Frontiers in Microbiology,25a655e4-17da-4bfc-a246-5cd20202068d,2023.0,47,13,1,True,"[{'category': 'Medicine', 'source': 's2-fos-mo...",['JournalArticle'],2023-07-04,"{'name': 'Frontiers in Microbiology', 'pages':..."
1,259015091,"{'MAG': None, 'CorpusId': '259015091', 'ACL': ...",https://www.semanticscholar.org/paper/fff2190c...,Advanced Applications of Carbonaceous Material...,"[{'authorId': '102987198', 'name': 'M. S. Reza...",Sustainability,8775599f-4f9a-45f0-900e-7f4de68e6843,2023.0,472,49,1,True,NaN,"['JournalArticle', 'Review']",2023-05-30,"{'name': 'Sustainability', 'pages': None, 'vol..."
2,256229419,"{'MAG': None, 'CorpusId': '256229419', 'ACL': ...",https://www.semanticscholar.org/paper/0ef70f09...,Characterisation of human milk bacterial DNA p...,"[{'authorId': '1431759814', 'name': 'Azhar S. ...",PLoS ONE,0aed7a40-85f3-4c66-9e1b-c1556c57001b,2023.0,81,7,0,True,"[{'category': 'Medicine', 'source': 's2-fos-mo...",['JournalArticle'],2023-01-25,"{'name': 'PLOS ONE', 'pages': None, 'volume': ..."
3,279719947,"{'MAG': None, 'CorpusId': '279719947', 'ACL': ...",https://www.semanticscholar.org/paper/e1ab88e0...,A control theory framework and in situ experim...,"[{'authorId': '2084584156', 'name': 'Jason L. ...",Nature Ecology & Evolution,7331e868-0ea2-450b-8ff6-106c6bc80786,2025.0,308,3,0,False,"[{'category': 'Environmental Science', 'source...","['Review', 'JournalArticle']",2025-06-30,"{'name': 'Nature Ecology & Evolution', 'pages'..."
4,265193413,"{'MAG': None, 'CorpusId': '265193413', 'ACL': ...",https://www.semanticscholar.org/paper/8f749d11...,Decision-making in closed-loop supply chains: ...,"[{'authorId': '2266768230', 'name': 'Pengyu Sh...",Journal of Cleaner Production,95308966-10a0-4b50-b00f-f130a37f7d3c,2023.0,28,6,0,False,NaN,['JournalArticle'],2023-11-01,"{'name': 'Journal of Cleaner Production', 'pag..."


In [11]:
paper_ids = [x.split("/")[-1] for x in df_all_papers["url"]]
print(f"{len(paper_ids):,} paper ids")

14,534 paper ids


In [12]:
json_path = release_path / "json"
json_path.mkdir(parents=True, exist_ok=True)

for i, group in tqdm(
    enumerate(s2_api.chunker(paper_ids, MAX_NUM_BULK_PAPERS)),
    total=int(len(paper_ids) / MAX_NUM_BULK_PAPERS),
):
    response_batch = s2_client.download_bulk_paper(group)

    json_file = json_path / f"{s2_client.latest_release}-{i}.json"
    with open(str(json_file), "w", encoding="utf-8") as f:
        json.dump(response_batch.json(), f)

  0%|          | 0/29 [00:00<?, ?it/s]

2026-06-04 02:11:41.245 | DEBUG    | kibad_crawler.s2_api:download_bulk_paper:386 - Bulk paper request: 500 IDs


  3%|▎         | 1/29 [00:04<02:03,  4.42s/it]

2026-06-04 02:11:45.669 | DEBUG    | kibad_crawler.s2_api:download_bulk_paper:386 - Bulk paper request: 500 IDs


  7%|▋         | 2/29 [00:08<01:51,  4.13s/it]

2026-06-04 02:11:49.597 | DEBUG    | kibad_crawler.s2_api:download_bulk_paper:386 - Bulk paper request: 500 IDs


... (51 outputs left out)

2026-06-04 02:13:38.943 | DEBUG    | kibad_crawler.s2_api:download_bulk_paper:386 - Bulk paper request: 500 IDs


100%|██████████| 29/29 [02:02<00:00,  4.30s/it]

2026-06-04 02:13:43.417 | DEBUG    | kibad_crawler.s2_api:download_bulk_paper:386 - Bulk paper request: 34 IDs


30it [02:03,  4.11s/it]                        


In [ ]:
def download_paper_batch(
    url: str,
    publication_venue_ids: list,
    persistant: bool = False,
    start_date: str = "2023-01-01",
) -> None:
    file_name = url.split("?")[0].split("/")[-1]
    if (release_path / "papers" / str(file_name + ".csv")).exists():
        return

    file_path = download_file(url=url, path=release_path / "papers")
    file_content = ""
    with gzip.open(
        file_path,
        "r",
    ) as f:
        for line in tqdm(f.readlines(), leave=True):
            if any(x in line.decode() for x in publication_venue_ids):
                file_content += line.decode()

    df_papers = pd.read_json(StringIO(file_content), lines=True)

    date_filter = datetime.strptime(start_date, "%Y-%m-%d")
    df_papers["publicationdate"] = pd.to_datetime(df_papers["publicationdate"])
    df_papers = df_papers[df_papers["publicationdate"] >= date_filter]

    df_papers.to_csv(release_path / "papers" / str(file_name + ".csv"), index=False)

    if not persistant:
        os.remove(file_path)


def download_all_papers(
    file_list: list, publication_venue_ids: list, persistant: bool = False
) -> None:
    for url in tqdm(file_list, total=len(file_list)):
        download_paper_batch(
            url=url, publication_venue_ids=publication_venue_ids, persistant=persistant
        )

In [ ]:
# TODO Add a max_date to filter
s2_api.download_all_papers(
    file_list=response_papers.json()["files"],
    release_path=release_path,
    publication_venue_ids=list(publication_venue_ids),
)

## Bulk paper download

In [ ]:
paper_files = [x for x in (release_path / "papers").glob("*.csv")]
dfs = [pd.read_csv(f) for f in paper_files]
df_all_papers = pd.concat(dfs)
print(f"{len(df_all_papers):,} papers in total")

In [9]:
MIN_NUMBER_OF_CITATIONS: int = 2

df_all_papers_citations = df_all_papers[df_all_papers["citationcount"] > MIN_NUMBER_OF_CITATIONS]

In [ ]:
paper_ids = [x.split("/")[-1] for x in df_all_papers_citations["url"]]
print(f"{len(paper_ids):,} paper ids")

8,217 paper ids


In [ ]:
json_path = release_path / "json"
json_path.mkdir(parents=True, exist_ok=True)

for i, group in tqdm(
    enumerate(s2_api.chunker(paper_ids[:2_000], MAX_NUM_BULK_PAPERS)),
    total=int(2_000 / MAX_NUM_BULK_PAPERS),
):
    response_batch = s2_api.download_bulk_paper(group)

    json_file = json_path / f"{release_id}-{i}.json"
    with open(str(json_file), "w", encoding="utf-8") as f:
        json.dump(response_batch.json(), f)

## Read json files

In [ ]:
paper_files = [x for x in (release_path / "json").glob("*.json")]
dfs = [pd.read_json(f) for f in paper_files]
df_all_papers = pd.concat(dfs)
print(f"{len(df_all_papers):,} json in total")

# Incremental Updates (WiP)

In [ ]:
response_releases = requests.get("https://api.semanticscholar.org/datasets/v1/release").json()
latest_release = response_releases[-1]
print(f"{response_releases[-1]} latest release")

In [ ]:
response_releases[-3:]

In [ ]:
release_path = RAW_DATA_DIR / "semantic_scholar" / "release" / f"{latest_release}"
release_path.mkdir(exist_ok=True, parents=True)

In [ ]:
from typing import Literal

URL_INCREMENTAL = "https://api.semanticscholar.org/datasets/v1/diffs/{start_release_id}/to/{end_release_id}/{dataset_name}"


def download_incremental(
    start_release_id: str,
    end_release_id: str,
    dataset_name: Literal[
        "abstracts",
        "authors",
        "citations",
        "embeddings-specter_v1",
        "embeddings-specter_v2",
        "paper-ids",
        "papers",
        "publication-venues",
        "s2orc",
        "s2orc_v2",
        "tldrs",
    ],
) -> requests.Response:
    headers = {"x-api-key": S2_API_KEY}
    response = requests.get(
        URL_INCREMENTAL.format(
            start_release_id=start_release_id,
            end_release_id=end_release_id,
            dataset_name=dataset_name,
        ),
        headers=headers,
        timeout=60,
    )
    return response


response = download_incremental(
    start_release_id="2026-01-27", end_release_id="2026-02-10", dataset_name="papers"
)

In [ ]:
# ['dataset', 'start_release', 'end_release', 'diffs']
len(response.json()["diffs"][0]["update_files"])

In [ ]:
response.json()["diffs"][0]["update_files"][0]